<div class="alert alert-block alert-danger">
Before you start running this notebook, create a Hail-enabled Dataproc cluster with a <b>high compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab Spark cluster</b> and, under <b>Software to install</b>, choose <b>Hail</b>.</li>
    <li>Configure a high-compute cluster (multiple workers). Cluster creation can take about 10 minutes.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a> and <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/cloud_app_types/dataproc/">Use Dataproc (Spark) and Hail</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook prepares Hail datasets for downstream short-read analyses of the Phase 1 long-read cohort. It identifies the 1,027 long-read participants and the ~989 of those who also have GATK-SV calls, then subsets the ~245k-sample v7 short-read Hail VDS to those samples and writes compact VDS and MatrixTable copies.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
import pandas
import os

In [ ]:
# Query the CDR for every participant with both short-read and long-read WGS (the 1,027-sample long-read cohort).
# The SQL was generated by the Researcher Workbench Dataset Builder.
# This query represents dataset "Long reads" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_44446189_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 ) )"""

dataset_44446189_person_df = pandas.read_gbq(
    dataset_44446189_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_44446189_person_df.head(5)

In [ ]:
import hail as hl

In [ ]:
# Start Hail on the notebook's Spark cluster, using GRCh38 coordinates.
hl.init(idempotent=True)

In [ ]:
hl.default_reference('GRCh38')

In [ ]:
# Workspace details the Workbench sets as environment variables.
bucket = os.environ['WORKSPACE_BUCKET']
workspace = os.environ['WORKSPACE_NAME']
namespace = os.environ['WORKSPACE_NAMESPACE']

In [ ]:
print(bucket)
print(workspace)
print(namespace)

In [ ]:
# List, then clear, any intermediate files left in the scratch area by a previous run.
!gsutil ls {bucket}/scratch/

In [ ]:
import os

# Location of this notebook's data (a gs:// location). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA = os.environ["AOU_LR_DATA"]

In [ ]:
!gsutil -m rm -rf {AOU_LR_DATA}/scratch/

In [ ]:
# The Controlled Tier short-read (srWGS) v7 SNV/indel callset, as a Hail VariantDataset.
srs_vds = hl.vds.read_vds('gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/vds/hail.vds')

In [ ]:
# Keep only the long-read cohort, drop alleles no longer seen in any kept sample, and save the subset.
srs_subset_1027_vds = hl.vds.filter_samples(
    srs_vds,
    list(dataset_44446189_person_df['person_id'].astype(str)),
    keep = True,
    remove_dead_alleles = True
)
srs_subset_1027_vds.write(f'{bucket}/scratch/srs-subset.1027_samples.vds')

In [ ]:
# Reload the saved subset and confirm the sample count.
srs_subset_1027_vds = hl.vds.read_vds(f'{bucket}/scratch/srs-subset.1027_samples.vds')

In [ ]:
srs_subset_1027_vds.n_samples()

In [ ]:
# Convert the sparse VDS to a dense MatrixTable (one entry per sample per site).
srs_subset_1027_mt = hl.vds.to_dense_mt(srs_subset_1027_vds)
srs_subset_1027_mt.write(f'{bucket}/scratch/srs-subset.1027_samples.mt')

In [ ]:
# The subset keeps the full callset's partition count, though it holds far less data. Repartition to
# (original partition count / number of samples) partitions.
srs_subset_1027_mt = hl.read_matrix_table(f'{bucket}/scratch/srs-subset.1027_samples.mt')
srs_subset_1027_mt = srs_subset_1027_mt.repartition(int(srs_subset_1027_mt.n_partitions() / 1027))
srs_subset_1027_mt.write(f'{bucket}/scratch/srs-subset.1027_samples.repartitioned.mt')

In [ ]:
srs_subset_1027_mt = hl.read_matrix_table(f'{bucket}/scratch/srs-subset.1027_samples.repartitioned.mt')

In [ ]:
# Query the subset of the long-read cohort that also has short-read SV calls (989 participants).
# This query represents dataset "Long reads with short read SVs, basic info" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_47029769_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_structural_variant_data = 1 ) )"""

dataset_47029769_person_df = pandas.read_gbq(
    dataset_47029769_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_47029769_person_df.head(5)

In [ ]:
# Repeat the subsetting, densifying, and repartitioning for the 989-sample cohort.
srs_subset_989_vds = hl.vds.filter_samples(
    srs_subset_1027_vds,
    list(dataset_47029769_person_df['person_id'].astype(str)),
    keep = True,
    remove_dead_alleles = True
)
srs_subset_989_vds.write(f'{bucket}/scratch/srs-subset.989_samples.vds')

In [ ]:
srs_subset_989_vds.n_samples()

In [ ]:
srs_subset_989_mt = hl.vds.to_dense_mt(srs_subset_989_vds)
srs_subset_989_mt.write(f'{bucket}/scratch/srs-subset.989_samples.mt')

In [ ]:
srs_subset_989_mt = hl.read_matrix_table(f'{bucket}/scratch/srs-subset.989_samples.mt')
srs_subset_989_mt = srs_subset_989_mt.repartition(int(srs_subset_989_mt.n_partitions() / 989))
srs_subset_989_mt.write(f'{bucket}/scratch/srs-subset.989_samples.repartitioned.mt')

In [ ]:
srs_subset_989_mt = hl.read_matrix_table(f'{bucket}/scratch/srs-subset.989_samples.repartitioned.mt')

In [ ]:
# Sample IDs in the final 989-sample MatrixTable.
srs_subset_989_mt.s.collect()